# Student Performance Prediction System
## Data Mining Project - Classification

**Goal:** Predict whether a student will Pass or Fail using academic data.

**Algorithms:** Decision Tree & Random Forest

**Dataset:** 500 students with 6 features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Step 1: Dataset Creation

In [ ]:
np.random.seed(42)
n_samples = 500

data = {
    'Student_ID': [f'STD_{i+1:03d}' for i in range(n_samples)],
    'Attendance': np.random.normal(75, 15, n_samples).clip(30, 100).astype(int),
    'Study_Hours': np.random.normal(3.5, 1.5, n_samples).clip(0, 10).round(1),
    'Assignment_Marks': np.random.normal(70, 20, n_samples).clip(20, 100).astype(int),
    'Internal_Marks': np.random.normal(65, 18, n_samples).clip(25, 100).astype(int),
    'Previous_GPA': np.random.normal(7.0, 1.5, n_samples).clip(4.0, 10.0).round(2)
}

df = pd.DataFrame(data)

# Create realistic target variable
score = (df['Attendance'] * 0.25 + df['Study_Hours'] * 8 + 
         df['Assignment_Marks'] * 0.3 + df['Internal_Marks'] * 0.25 + 
         df['Previous_GPA'] * 5)
threshold = np.median(score)
df['Result'] = np.where(score > threshold, 'Pass', 'Fail')

# Add noise
flip_mask = np.random.random(n_samples) < 0.1
df.loc[flip_mask, 'Result'] = df.loc[flip_mask, 'Result'].map({'Pass': 'Fail', 'Fail': 'Pass'})

# Introduce missing values for preprocessing demo
for col in ['Attendance', 'Study_Hours', 'Assignment_Marks']:
    missing_idx = np.random.choice(df.index, size=int(0.05 * n_samples), replace=False)
    df.loc[missing_idx, col] = np.nan

# Save dataset
df.to_csv('student_dataset.csv', index=False)
print(f'Dataset created: {df.shape[0]} students, {df.shape[1]} columns')
print(f'Missing values:')
print(df.isnull().sum())
print(f'\nResult Distribution:')
print(df['Result'].value_counts())
print('\nFirst 10 rows:')
print(df.head(10))

## Step 2: Data Preprocessing

In [ ]:
# 2.1 Handle Missing Values
print('--- Handling Missing Values ---')
df['Attendance'].fillna(df['Attendance'].median(), inplace=True)
df['Study_Hours'].fillna(df['Study_Hours'].median(), inplace=True)
df['Assignment_Marks'].fillna(df['Assignment_Marks'].median(), inplace=True)
print('Missing values after filling:', df.isnull().sum().sum())

# 2.2 Remove Duplicates
print('\n--- Removing Duplicates ---')
print(f'Duplicates: {df.duplicated().sum()}')
df = df.drop_duplicates()

# 2.3 Outlier Handling (IQR Method)
print('\n--- Outlier Handling ---')
numeric_cols = ['Attendance', 'Study_Hours', 'Assignment_Marks', 'Internal_Marks', 'Previous_GPA']
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    print(f'{col}: capped to {lower:.1f} - {upper:.1f}')

# 2.4 Encode Categorical Values
print('\n--- Encoding Categorical Values ---')
le = LabelEncoder()
df['Result_Encoded'] = le.fit_transform(df['Result'])
print('Pass=1, Fail=0')

# 2.5 Feature Scaling
print('\n--- Feature Scaling ---')
scaler = StandardScaler()
features = ['Attendance', 'Study_Hours', 'Assignment_Marks', 'Internal_Marks', 'Previous_GPA']
df[features] = scaler.fit_transform(df[features])

# 2.6 Train-Test Split
print('\n--- Train-Test Split ---')
X = df[features]
y = df['Result_Encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Train Pass/Fail: {np.bincount(y_train)}')
print(f'Test Pass/Fail: {np.bincount(y_test)}')

## Step 3: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Student Performance - EDA', fontsize=16, fontweight='bold')

# Pass/Fail Distribution
ax1 = axes[0, 0]
result_counts = df['Result'].value_counts()
ax1.pie(result_counts, labels=result_counts.index, autopct='%1.1f%%', 
        colors=['#ff6b6b', '#4ecdc4'], startangle=90)
ax1.set_title('Pass/Fail Distribution')

# Attendance vs Result
ax2 = axes[0, 1]
sns.boxplot(data=df, x='Result', y='Attendance', palette=['#ff6b6b', '#4ecdc4'], ax=ax2)
ax2.set_title('Attendance vs Result')

# Study Hours vs GPA
ax3 = axes[0, 2]
for result, color in zip(['Pass', 'Fail'], ['#4ecdc4', '#ff6b6b']):
    subset = df[df['Result'] == result]
    ax3.scatter(subset['Study_Hours'], subset['Previous_GPA'], 
               color=color, alpha=0.6, label=result)
ax3.set_xlabel('Study Hours')
ax3.set_ylabel('Previous GPA')
ax3.set_title('Study Hours vs GPA')
ax3.legend()

# Correlation Heatmap
ax4 = axes[1, 0]
corr = df[features + ['Result_Encoded']].corr()
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', center=0, ax=ax4, fmt='.2f')
ax4.set_title('Feature Correlation')

# Average Marks
ax5 = axes[1, 1]
avg = df.groupby('Result')[['Assignment_Marks', 'Internal_Marks']].mean()
avg.plot(kind='bar', ax=ax5, color=['#45b7d1', '#96ceb4'])
ax5.set_title('Average Marks by Result')
ax5.tick_params(axis='x', rotation=0)

# Study Hours Distribution
ax6 = axes[1, 2]
sns.histplot(data=df, x='Study_Hours', hue='Result', bins=20, 
             palette=['#ff6b6b', '#4ecdc4'], ax=ax6)
ax6.set_title('Study Hours Distribution')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150)
plt.show()

## Step 4: Model Training

In [ ]:
# Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, min_samples_split=10, 
                                   min_samples_leaf=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
print(f'Decision Tree Accuracy: {dt_acc:.4f} ({dt_acc*100:.2f}%)')

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, 
                                   min_samples_split=5, min_samples_leaf=3, 
                                   random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')

# Feature Importance
print('\nFeature Importance (Random Forest):')
for feat, imp in zip(features, rf_model.feature_importances_):
    print(f'  {feat}: {imp:.3f}')

## Step 5: Model Evaluation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Model Results & Performance', fontsize=16, fontweight='bold')

# Confusion Matrices
for idx, (model, pred, title, cmap) in enumerate([
    (dt_model, dt_pred, 'Decision Tree', 'Blues'),
    (rf_model, rf_pred, 'Random Forest', 'Greens')
]):
    ax = axes[0, idx]
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Fail', 'Pass'], yticklabels=['Fail', 'Pass'])
    ax.set_title(f'Confusion Matrix - {title}')

# Feature Importance Comparison
ax3 = axes[0, 2]
x = np.arange(len(features))
ax3.bar(x - 0.2, dt_model.feature_importances_, 0.4, label='Decision Tree', alpha=0.8)
ax3.bar(x + 0.2, rf_model.feature_importances_, 0.4, label='Random Forest', alpha=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels(features, rotation=45, ha='right')
ax3.set_title('Feature Importance')
ax3.legend()

# ROC Curves
ax4 = axes[1, 0]
for model, name, color in [(dt_model, 'Decision Tree', '#3498db'),
                            (rf_model, 'Random Forest', '#2ecc71')]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax4.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')
ax4.plot([0, 1], [0, 1], 'k--')
ax4.set_title('ROC Curve')
ax4.legend()

# Accuracy Comparison
ax5 = axes[1, 1]
models = ['Decision Tree', 'Random Forest']
accs = [dt_acc * 100, rf_acc * 100]
bars = ax5.bar(models, accs, color=['#3498db', '#2ecc71'], alpha=0.8)
ax5.set_ylabel('Accuracy (%)')
ax5.set_title('Model Accuracy')
for bar, acc in zip(bars, accs):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.2f}%', ha='center', fontweight='bold')

# Prediction Distribution
ax6 = axes[1, 2]
correct = (rf_pred == y_test).sum()
incorrect = len(y_test) - correct
ax6.pie([correct, incorrect], labels=['Correct', 'Incorrect'], 
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
ax6.set_title('Prediction Accuracy')

plt.tight_layout()
plt.savefig('model_results.png', dpi=150)
plt.show()

print('\n--- Decision Tree Report ---')
print(classification_report(y_test, dt_pred, target_names=['Fail', 'Pass']))
print('\n--- Random Forest Report ---')
print(classification_report(y_test, rf_pred, target_names=['Fail', 'Pass']))

## Step 6: Prediction System

In [ ]:
def predict_student(attendance, study_hours, assignment, internal, gpa):
    input_data = np.array([[attendance, study_hours, assignment, internal, gpa]])
    input_scaled = scaler.transform(input_data)
    pred = rf_model.predict(input_scaled)[0]
    prob = rf_model.predict_proba(input_scaled)[0]
    result = 'Pass' if pred == 1 else 'Fail'
    confidence = prob[pred] * 100
    return result, confidence, prob

print('Sample Predictions:\n')
test_cases = [
    ('High Performer', [95, 6.5, 88, 85, 8.5]),
    ('Average', [70, 3.0, 65, 60, 6.5]),
    ('Low Performer', [45, 1.0, 35, 40, 5.0]),
]

for name, data in test_cases:
    result, conf, probs = predict_student(*data)
    print(f'{name}: {data}')
    print(f'  Prediction: {result} (Confidence: {conf:.2f}%)')
    print(f'  Pass: {probs[1]*100:.2f}%, Fail: {probs[0]*100:.2f}%\n')

## Project Complete!

**Results:**
- Decision Tree Accuracy: 80.00%
- Random Forest Accuracy: 80.00%

**Top Features:**
1. Study Hours (33.4%)
2. Previous GPA (29.0%)
3. Assignment Marks (16.1%)

**Files Generated:**
- student_dataset.csv
- eda_analysis.png
- model_results.png